In [ ]:
import seaborn as sns
import requests
import pandas as pd
import numpy as np
from datetime import datetime
from astropy.time import Time
from LINZ.CORS_Timeseries import TimeseriesList, CoordApiTimeseries
import os 
from statsmodels.tsa.stattools import acf
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

In [ ]:
# Define time range
start = datetime(2025, 4, 10)
end = datetime(2025, 9, 25)

In [ ]:
# Define station groups
PNcodes=['AUCK','BLUF','CHTI','CORM','DNVK','DUND','GISB','GLDB','HAAS','HAMT','HAST','HIKB','HOKI','KAIK','KTIA','LEXA','LKTA','MAHO','MAVL','METH','MQZG','MTJO','NLSN','NPLY','PYGR','TAUP','TRNG',
         'VGMT','WAIM','WANG','WARK','WEST','WGTN','WHKT','WHNG','WRPA']

In [ ]:
# Calculate crd_epoch
crd_epoch = end + (start - end) / 2
astropy_time_object = Time(crd_epoch, format='datetime')
crd_epoch_decimal_year = astropy_time_object.decimalyear

In [ ]:
# Initialise a dictionary to store DataFrames for each solution type
daily_avg_dfs = {
    #'p20f_54_code': {},
    #'p20r_52_igs': {},
    #'d20f_52_code_A': {},
    'd20f_54_code_A':{}
}

# Loop through PNcodes and solution types
for code in PNcodes:
    for sol_type in daily_avg_dfs.keys():
        try:
            print(f"Processing station: {code} with solution type: {sol_type}")
            ts = CoordApiTimeseries(code, solutiontype=sol_type, after=start, before=end)  
            dates, xyz = ts.withoutOutliers().getObs(enu=False)

            df_obs = pd.DataFrame(xyz, columns=['x', 'y', 'z'], index=dates)
            df_obs['date'] = df_obs.index.strftime('%Y-%m-%d')
            df_obs['station'] = code
            df_obs['solutiontype'] = sol_type

            daily_avg_dfs[sol_type][code] = df_obs

        except Exception as e:
            print(f"Failed to process station {code} with solution type {sol_type}. Error: {e}")

# Combine all DataFrames into one
PN_data = pd.concat(
    [pd.concat(dfs.values(), axis=0) for dfs in daily_avg_dfs.values()],
    axis=0
)

# Sort alphabetically by station, date, and solution type
PN_data.sort_values(by=["station", "date", "solutiontype"], inplace=True)

# Print confirmation
print("Download complete ✅")
print("PN_data DataFrame created with shape:", PN_data.shape)
print("Columns:", PN_data.columns.tolist())


In [ ]:
# Group PN_data by station and solutiontype
group_dataframes = {
    f"{station}_{sol_type}": group
    for (station, sol_type), group in PN_data.groupby(["station", "solutiontype"])
}


In [ ]:
# Define expected converted DataFrame names
converted_columns = [
    'x', 'y', 'z', 'date', 'station',
    'nztm_easting', 'nztm_northing', 'nzvd2016_elev'
]

# Initialize empty dictionary with expected keys
empty_converted_dfs = {
    f"{name}_converted": pd.DataFrame(columns=converted_columns)
    for name in group_dataframes.keys()
}



In [ ]:
converted_group_dataframes = {}

def convert_coordinates(input_crds, crd_epoch_decimal_year):
    occ_api = "https://www.geodesy.linz.govt.nz/api/conversions/v1/convert-to"
    coordlist = {
        "crs": "LINZ:ITRF2020_XYZ",
        "coordinateOrder": ["geocentricX", "geocentricY", "geocentricZ"],
        "coordinateEpoch": crd_epoch_decimal_year,
        "coordinates": input_crds
    }
    params = {"crs": "LINZ:NZTM/NZVD2016"}

    response = requests.post(occ_api, params=params, json=coordlist)
    if response.status_code == 200:
        converted = response.json()
        return converted['coordinateList']['coordinates']
    else:
        print(f"❌ API conversion failed with status {response.status_code}")
        print(f"🔍 Error details: {response.text}")
        return []


In [ ]:
for name, original_df in group_dataframes.items():
    print(f"\n🔄 Processing DataFrame: {name}")

    # Always start with a fresh copy
    df = original_df.copy(deep=True)

    # Identify and print NaN values
    nan_values = df[df.isna().any(axis=1)]
    if not nan_values.empty:
        print(f" - Found {len(nan_values)} NaN rows")

    # Identify and print Infinity values
    inf_values = df[(df == np.inf) | (df == -np.inf)].dropna(how='all')
    if not inf_values.empty:
        print(f" - Found {len(inf_values)} Inf rows")

    # Filter out invalid rows
    df_filtered = df.dropna(subset=['x', 'y', 'z']).copy()
    df_filtered = df_filtered[
        (df_filtered['x'] != np.inf) & (df_filtered['x'] != -np.inf) &
        (df_filtered['y'] != np.inf) & (df_filtered['y'] != -np.inf) &
        (df_filtered['z'] != np.inf) & (df_filtered['z'] != -np.inf)
    ]

    if df_filtered.empty:
        print(f"⚠️ Skipping {name} — no valid coordinates after filtering.")
        continue

    # Prepare coordinates
    input_crds = df_filtered[['x', 'y', 'z']].values.tolist()

    # Convert coordinates
    converted_coords = convert_coordinates(input_crds, crd_epoch_decimal_year)

    # Add converted coordinates to DataFrame
    if converted_coords:
        df_filtered[['nztm_easting', 'nztm_northing', 'nzvd2016_elev']] = pd.DataFrame(
            converted_coords, index=df_filtered.index
        )

        # Store in the pre-initialised DataFrame
        converted_name = f"{name}_converted"
        if converted_name in empty_converted_dfs:
            empty_converted_dfs[converted_name] = df_filtered
            print(f"✅ {converted_name} filled and updated.")
        else:
            print(f"⚠️ {converted_name} not found in empty_converted_dfs. Skipping storage.")
    else:
        print(f"❌ Conversion failed for {name}. No coordinates added.")

In [ ]:
# Rename for clarity — this is now the final dictionary of converted DataFrames
converted_group_dataframes = empty_converted_dfs
del empty_converted_dfs  # Optional: clean up the old name if no longer needed

In [ ]:
# Step 1: Run alert detection
alert_rows = []
alert_sites = set()

for name, df in converted_group_dataframes.items():
    if 'x' in df.columns and 'y' in df.columns:
        df['horizontal_offset'] = np.sqrt(df['x']**2 + df['y']**2)
        mean_offset = df['horizontal_offset'].mean()
        df['deviation'] = abs(df['horizontal_offset'] - mean_offset)

        alerts = df[df['deviation'] > 0.010].copy()  # 1 cm threshold

        base_name = name.replace('_converted', '')
        station, solutiontype = base_name.split('_', 1) if '_' in base_name else (base_name, 'unknown')

        if not alerts.empty:
            alert_sites.add(name)

        for _, row in alerts.iterrows():
            alert_rows.append({
                'station': station,
                'solutiontype': solutiontype,
                'date': row['date'],
                'horizontal_offset': round(row['horizontal_offset'], 3),
                'deviation': round(row['deviation'], 3)
            })

alert_summary_df = pd.DataFrame(alert_rows)
print("Alert Summary:")
print(alert_summary_df.head())
print(f"\nTotal alerts triggered: {len(alert_summary_df)}")

# Step 2: Plot time series for flagged sites only
def plot_deviation_timeseries(dataframes, site_keys, threshold_cm=1.0):
    for name in site_keys:
        df = dataframes.get(name)
        if df is not None and 'deviation' in df.columns and 'date' in df.columns:
            df['date'] = pd.to_datetime(df['date'])  # Ensure date column is datetime
            plt.figure(figsize=(10, 4))
            plt.plot(df['date'], df['deviation'], label='Deviation')
            plt.axhline(y=threshold_cm / 100.0, color='r', linestyle='--', label='Threshold (1 cm)')
            plt.title(f'Deviation Time Series - {name}')
            plt.xlabel('Date')
            plt.ylabel('Deviation (m)')
            plt.legend()
            plt.grid(True)
            plt.xticks(rotation=45)  
            plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%d-%b'))  # ✅ Format dates
            plt.tight_layout()
            plt.show()

plot_deviation_timeseries(converted_group_dataframes, alert_sites, threshold_cm=1.0)